### New predictions for selected model (Global Model 1959-2024)

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# predict_global_yearly.py
#
# Resume-safe global prediction with xgbGlobal_sigHail
# - loads model once
# - loads predictor grid + LSM alignment once
# - predicts year by year
# - skips existing yearly outputs
# - skips missing predictor files
# - saves:
#     * daily probabilities
#     * daily binary predictions
#     * yearly mean predicted hail frequency
#
# Notes
# - applies the same predictor selection as training
# - applies abs() transform to VS03 and VS06 as in training
# - masks ocean before prediction
# - predicts in time chunks to avoid memory issues
# ============================================================

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from netCDF4 import Dataset
import xgboost as xgb


# ============================================================
# 1) CONFIG
# ============================================================
cfg = {
    "start_day": datetime(1959, 1, 1, 0),
    "stop_day": datetime(2024, 12, 31, 23),
    "missing_years": [1966, 1979, 1999, 2003],
    "fold_to_use": 0,
    "thr_bin": 0.5,
    "chunk_days": 60,  # safer for global runs than 180
    "save_daily_prob": False,   # set True if you really want daily probabilities
    "save_daily_bin": False,    # set True if you really want daily binary fields
}

base_cumulus = Path("/nfs/cumulus/highres_nobackup/agebhardt")
base_cluster = Path("/cluster/home/agebhardt")

run_date = "xgbGlobal_sigHail/final_dall"

paths = {
    "era_global": base_cumulus / "e5_hailpredictors",
    "era_invar": base_cumulus / "e5_data_processing" / "e5_invariant",
    "models": base_cluster / "models" / run_date,
    "out": base_cumulus / "preds_1959-2024_xgbGlobal/preds_global/fold0_dall"
}
paths["out"].mkdir(parents=True, exist_ok=True)

files = {
    "const_fields": paths["era_invar"] / "e5_invariant_129_z_ll025sc.2020010100_2020010100.nc",
    # "lsm": paths["era_invar"] / "e5_lsm_11024sc.nc",
    "model": paths["models"] / f"xgbGlobal_sigHail_FINAL_2000-2022_fold{cfg['fold_to_use']}.json",
}

all_model_vars = [
    "CAPEmax", "SRH03", "VS03", "FLH",
    "CINmax", "SRH06", "VS06", "DewT",
    "TotalTotals", "RH850", "RH500",
]
model_vars = [
    "CAPEmax", "SRH03", "VS03", "FLH",
    "CINmax", "VS06", "DewT",
    "TotalTotals",
]
sel_idx = [all_model_vars.index(v) for v in model_vars]
ix_vs03 = model_vars.index("VS03")
ix_vs06 = model_vars.index("VS06")


# ============================================================
# 2) HELPERS
# ============================================================
# def to_minus180_180(lon_1d):
#     lon_1d = np.asarray(lon_1d).copy()
#     return np.where(lon_1d > 180, lon_1d - 360, lon_1d)


# def build_index_map(src_lat, src_lon, tgt_lat, tgt_lon, tol=1e-6):
#     src_lat = np.asarray(src_lat)
#     src_lon = np.asarray(src_lon)
#     tgt_lat = np.asarray(tgt_lat)
#     tgt_lon = np.asarray(tgt_lon)

#     ilat = np.empty(tgt_lat.size, dtype=int)
#     for i, val in enumerate(tgt_lat):
#         hits = np.where(np.isclose(src_lat, val, atol=tol, rtol=0))[0]
#         if hits.size == 0:
#             raise RuntimeError(f"Could not match latitude {val} within tol={tol}.")
#         ilat[i] = int(hits[0])

#     ilon = np.empty(tgt_lon.size, dtype=int)
#     for i, val in enumerate(tgt_lon):
#         hits = np.where(np.isclose(src_lon, val, atol=tol, rtol=0))[0]
#         if hits.size == 0:
#             raise RuntimeError(f"Could not match longitude {val} within tol={tol}.")
#         ilon[i] = int(hits[0])

#     return ilat, ilon


def get_years_to_process():
    dates = pd.date_range(cfg["start_day"], end=cfg["stop_day"], freq="D")
    dates = dates[~dates.year.isin(cfg["missing_years"])]
    years = np.unique(dates.year)
    return dates, years


def load_sample_predictor(years):
    for year in years:
        infile = paths["era_global"] / f"ERA5_new_predictors_{int(year)}.npz"
        if infile.is_file():
            data = np.load(infile)
            return int(year), data
    raise FileNotFoundError("Could not find any global predictor file.")


def ensure_2d_latlon(lat, lon):
    if lat.ndim == 1 and lon.ndim == 1:
        lon2d = np.tile(lon[None, :], (lat.size, 1))
        lat2d = np.tile(lat[:, None], (1, lon.size))
        return lat2d, lon2d
    return lat, lon


# def align_lsm_to_predictor_grid(lsm_file, lat_pred_1d, lon_pred_1d, ny, nx):
#     with Dataset(lsm_file, mode="r") as nc:
#         lsm = np.squeeze(nc.variables["lsm"][:]).astype(np.float32)
#         lat_lsm = np.asarray(nc.variables["latitude"][:]).astype(np.float32)
#         lon_lsm = np.asarray(nc.variables["longitude"][:]).astype(np.float32)

#     if lsm.ndim != 2:
#         raise RuntimeError(f"Expected 2D lsm, got shape {lsm.shape}")

#     if lsm.shape == (ny, nx):
#         print("LSM already matches predictor grid:", lsm.shape)
#         return lsm

#     lon_pred_match = np.asarray(lon_pred_1d).astype(np.float32)
#     lon_lsm_match = lon_lsm.astype(np.float32)

#     pred_is_0360 = (np.nanmin(lon_pred_match) >= 0.0) and (np.nanmax(lon_pred_match) > 180.0)
#     lsm_is_0360 = (np.nanmin(lon_lsm_match) >= 0.0) and (np.nanmax(lon_lsm_match) > 180.0)

#     if pred_is_0360 or lsm_is_0360:
#         lon_pred_match = to_minus180_180(lon_pred_match)
#         lon_lsm_match = to_minus180_180(lon_lsm_match)

#     ilat, ilon = build_index_map(
#         src_lat=lat_lsm,
#         src_lon=lon_lsm_match,
#         tgt_lat=lat_pred_1d,
#         tgt_lon=lon_pred_match,
#         tol=1e-6,
#     )
#     lsm_aligned = lsm[ilat][:, ilon]

#     if lsm_aligned.shape != (ny, nx):
#         raise RuntimeError(f"Aligned LSM shape {lsm_aligned.shape} != expected {(ny, nx)}")

#     print("LSM aligned from", lsm.shape, "to", lsm_aligned.shape)
#     return lsm_aligned


def load_model(model_path):
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")
    booster = xgb.Booster()
    booster.load_model(str(model_path))
    print("Loaded model:", model_path.name)
    return booster


def predict_year_chunked(booster, x_grid, chunk_days):
    """
    x_grid: (T, F, H, W)
    returns: y_prob (T, H, W)
    """
    t_size, n_feat, n_lat, n_lon = x_grid.shape
    y_prob = np.full((t_size, n_lat, n_lon), np.nan, dtype=np.float32)

    for t0 in range(0, t_size, chunk_days):
        t1 = min(t0 + chunk_days, t_size)
        x_chunk = np.moveaxis(x_grid[t0:t1], 1, -1)  # (T, H, W, F)
        tc = x_chunk.shape[0]

        x_flat = x_chunk.reshape(tc * n_lat * n_lon, n_feat)
        dmat = xgb.DMatrix(x_flat, missing=np.nan, feature_names=list(model_vars))
        pred = booster.predict(dmat).astype(np.float32).reshape(tc, n_lat, n_lon)

        y_prob[t0:t1] = pred
        print(f"    chunk {t0:03d}:{t1:03d} done")

    return y_prob


def prepare_predictors(rgr_era_vars):
    """
    rgr_era_vars: (T, F, H, W)
    """
    x_grid = rgr_era_vars.copy()

    # same transformations as in training
    x_grid[:, ix_vs03, :, :] = np.abs(x_grid[:, ix_vs03, :, :])
    x_grid[:, ix_vs06, :, :] = np.abs(x_grid[:, ix_vs06, :, :])

    return x_grid


def save_year_output(out_npz, year, y_prob, y_bin, mean_annual):
    save_dict = {
        "year": int(year),
        "mean_annual": mean_annual.astype(np.float32),
        "model_vars": np.array(model_vars, dtype="U"),
        "threshold_binary": np.float32(cfg["thr_bin"]),
        "fold": np.int32(cfg["fold_to_use"]),
        "model_path": str(files["model"]),
    }

    if cfg["save_daily_prob"]:
        save_dict["y_prob"] = y_prob.astype(np.float32)
    if cfg["save_daily_bin"]:
        save_dict["y_bin"] = y_bin.astype(np.float32)

    np.savez_compressed(out_npz, **save_dict)


# ============================================================
# 3) INITIAL SETUP
# ============================================================
print("Selected predictor indices:", sel_idx, "=>", [all_model_vars[i] for i in sel_idx])

full_dates, years_all = get_years_to_process()
print(
    "Years to process:",
    int(years_all.min()), "to", int(years_all.max()),
    "| missing:", cfg["missing_years"],
)

sample_year, sample = load_sample_predictor(years_all)
sample_vars = sample["rgrERAVarsyy"].astype(np.float32)[:, sel_idx, :, :]
lat_sample = sample.get("rgrLat", None)
lon_sample = sample.get("rgrLon", None)

if lat_sample is None or lon_sample is None:
    raise RuntimeError("Predictor .npz must contain rgrLat and rgrLon for global prediction.")

lat2d, lon2d = ensure_2d_latlon(lat_sample, lon_sample)
lat_pred_1d = lat2d[:, 0]
lon_pred_1d = lon2d[0, :]

_, _, ny, nx = sample_vars.shape
print(f"Sample year {sample_year} predictor shape: {sample_vars.shape} (T, F, H, W)")
print(f"Derived grid: ny={ny}, nx={nx}")

# lsm_global = align_lsm_to_predictor_grid(
#     lsm_file=files["lsm"],
#     lat_pred_1d=lat_pred_1d,
#     lon_pred_1d=lon_pred_1d,
#     ny=ny,
#     nx=nx,
# )
# print("Global land fraction:", float(np.mean(lsm_global >= 0.5)))

booster = load_model(files["model"])


# ============================================================
# 4) MAIN LOOP
# ============================================================
n_done = 0
n_skip_existing = 0
n_skip_missing_pred = 0
n_skip_grid_mismatch = 0

for year in years_all:
    year = int(year)

    infile = paths["era_global"] / f"ERA5_new_predictors_{year}.npz"
    out_npz = paths["out"] / f"pred_global_{year}_fold{cfg['fold_to_use']}.npz"

    if out_npz.is_file():
        print(f"[{year}] output exists -> skip")
        n_skip_existing += 1
        continue

    if not infile.is_file():
        print(f"[{year}] predictor missing -> skip")
        n_skip_missing_pred += 1
        continue

    print(f"\n[{year}] loading predictors")
    data_tmp = np.load(infile)
    rgr_era_vars_all = data_tmp["rgrERAVarsyy"].astype(np.float32)
    rgr_era_vars = rgr_era_vars_all[:, sel_idx, :, :]

    if rgr_era_vars.shape[2:] != (ny, nx):
        print(f"[{year}] grid mismatch -> skip. Got {rgr_era_vars.shape[2:]}, expected {(ny, nx)}")
        n_skip_grid_mismatch += 1
        continue

    x_grid = prepare_predictors(rgr_era_vars)

    print(f"[{year}] predicting")
    y_prob = predict_year_chunked(
        booster=booster,
        x_grid=x_grid,
        chunk_days=cfg["chunk_days"],
    )

    y_bin = (y_prob >= cfg["thr_bin"]).astype(np.float32)
    y_bin[np.isnan(y_prob)] = np.nan

    # mean annual predicted hail frequency per grid cell
    mean_annual = np.nansum(y_bin, axis=0) / y_bin.shape[0]

    save_year_output(
        out_npz=out_npz,
        year=year,
        y_prob=y_prob,
        y_bin=y_bin,
        mean_annual=mean_annual,
    )

    print(f"[{year}] saved: {out_npz.name} | daily={y_prob.shape} | annual={mean_annual.shape}")
    n_done += 1


# ============================================================
# 5) SUMMARY
# ============================================================
print("\nDONE.")
print("  years predicted:", n_done)
print("  skipped (output exists):", n_skip_existing)
print("  skipped (predictor missing):", n_skip_missing_pred)
print("  skipped (grid mismatch):", n_skip_grid_mismatch)
print("  output directory:", paths["out"])

Selected predictor indices: [0, 1, 2, 3, 4, 6, 7, 8] => ['CAPEmax', 'SRH03', 'VS03', 'FLH', 'CINmax', 'VS06', 'DewT', 'TotalTotals']
Years to process: 1959 to 2024 | missing: [1966, 1979, 1999, 2003]
Sample year 1959 predictor shape: (365, 8, 721, 1440) (T, F, H, W)
Derived grid: ny=721, nx=1440
Loaded model: xgbGlobal_sigHail_FINAL_2000-2022_fold0.json
[1959] output exists -> skip
[1960] output exists -> skip
[1961] output exists -> skip
[1962] output exists -> skip
[1963] output exists -> skip
[1964] output exists -> skip
[1965] output exists -> skip
[1967] output exists -> skip
[1968] output exists -> skip
[1969] output exists -> skip
[1970] output exists -> skip
[1971] output exists -> skip
[1972] output exists -> skip
[1973] output exists -> skip
[1974] output exists -> skip
[1975] output exists -> skip
[1976] output exists -> skip
[1977] output exists -> skip
[1978] output exists -> skip
[1980] output exists -> skip
[1981] output exists -> skip
[1982] output exists -> skip
[1983]

### New predictions for selected model (CONUS and Europe Model 1959-2024)

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# predict_region_yearly.py
#
# Resume-safe yearly prediction for CONUS or EU using xgbGlobal_sigHail
# - loads model once
# - reads already subsetted predictor files:
#     * e5_hailpredictors_conus
#     * e5_hailpredictors_eu
# - predicts year by year
# - skips existing yearly outputs
# - skips missing predictor files
# - saves:
#     * daily probabilities (optional)
#     * daily binary predictions (optional)
#     * yearly mean predicted hail frequency
#
# Notes
# - applies the same predictor selection as training
# - applies abs() transform to VS03 and VS06 as in training
# - NO land-sea mask is applied
# ============================================================

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import xgboost as xgb


# ============================================================
# 1) CONFIG
# ============================================================
cfg = {
    "start_day": datetime(1959, 1, 1, 0),
    "stop_day": datetime(2024, 12, 31, 23),
    "missing_years": [1966, 1979, 1999, 2003],
    "fold_to_use": 0,
    "thr_bin": 0.5,
    "chunk_days": 180,
    "save_daily_prob": False,
    "save_daily_bin": False,

    # choose one: "US" or "EU"
    "region": "EU",
}

base_cumulus = Path("/nfs/cumulus/highres_nobackup/agebhardt")
base_cluster = Path("/cluster/home/agebhardt")

run_date = "xgbGlobal_sigHail/final_dall"

region_cfg = {
    "US": {
        "era_dir": base_cumulus / "e5_hailpredictors_conus",
        "out_dir": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_US" / "fold0_dall",
        "pred_prefix": "pred_US",
        "infile_prefix": "ERA5_new_predictors_conus",
    },
    "EU": {
        "era_dir": base_cumulus / "e5_hailpredictors_eu",
        "out_dir": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_EU" / "fold0_dall",
        "pred_prefix": "pred_EU",
        "infile_prefix": "ERA5_new_predictors_eu",
    },
}

if cfg["region"] not in region_cfg:
    raise ValueError(f"Unknown region {cfg['region']}. Use 'US' or 'EU'.")

paths = {
    "era_region": region_cfg[cfg["region"]]["era_dir"],
    "models": base_cluster / "models" / run_date,
    "out": region_cfg[cfg["region"]]["out_dir"],
}
paths["out"].mkdir(parents=True, exist_ok=True)

files = {
    "model": paths["models"] / f"xgbGlobal_sigHail_FINAL_2000-2022_fold{cfg['fold_to_use']}.json",
}

all_model_vars = [
    "CAPEmax", "SRH03", "VS03", "FLH",
    "CINmax", "SRH06", "VS06", "DewT",
    "TotalTotals", "RH850", "RH500",
]

# selected predictors used by xgbGlobal
model_vars = [
    "CAPEmax", "SRH03", "VS03", "FLH",
    "CINmax", "VS06", "DewT",
    "TotalTotals",
]

sel_idx = [all_model_vars.index(v) for v in model_vars]
ix_vs03 = model_vars.index("VS03")
ix_vs06 = model_vars.index("VS06")


# ============================================================
# 2) HELPERS
# ============================================================
def get_years_to_process():
    dates = pd.date_range(cfg["start_day"], end=cfg["stop_day"], freq="D")
    dates = dates[~dates.year.isin(cfg["missing_years"])]
    years = np.unique(dates.year)
    return dates, years


def load_sample_predictor(years):
    infile_prefix = region_cfg[cfg["region"]]["infile_prefix"]

    for year in years:
        infile = paths["era_region"] / f"{infile_prefix}_{int(year)}.npz"
        if infile.is_file():
            data = np.load(infile)
            return int(year), data

    raise FileNotFoundError(
        f"Could not find any predictor file in {paths['era_region']}"
    )


def ensure_2d_latlon(lat, lon):
    lat = np.asarray(lat)
    lon = np.asarray(lon)

    if lat.ndim == 1 and lon.ndim == 1:
        lon2d = np.tile(lon[None, :], (lat.size, 1))
        lat2d = np.tile(lat[:, None], (1, lon.size))
        return lat2d, lon2d

    if lat.ndim == 2 and lon.ndim == 2:
        return lat, lon

    raise RuntimeError(f"Unexpected lat/lon shapes: lat{lat.shape}, lon{lon.shape}")


def load_model(model_path):
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")

    booster = xgb.Booster()
    booster.load_model(str(model_path))
    print("Loaded model:", model_path.name)
    return booster


def predict_year_chunked(booster, x_grid, chunk_days):
    """
    x_grid: (T, F, H, W)
    returns: y_prob (T, H, W)
    """
    t_size, n_feat, n_lat, n_lon = x_grid.shape
    y_prob = np.full((t_size, n_lat, n_lon), np.nan, dtype=np.float32)

    for t0 in range(0, t_size, chunk_days):
        t1 = min(t0 + chunk_days, t_size)
        x_chunk = np.moveaxis(x_grid[t0:t1], 1, -1)   # (T, H, W, F)
        tc = x_chunk.shape[0]

        x_flat = x_chunk.reshape(tc * n_lat * n_lon, n_feat)
        dmat = xgb.DMatrix(x_flat, missing=np.nan, feature_names=list(model_vars))
        pred = booster.predict(dmat).astype(np.float32).reshape(tc, n_lat, n_lon)

        y_prob[t0:t1] = pred
        print(f"    chunk {t0:03d}:{t1:03d} done")

    return y_prob


def prepare_predictors(rgr_era_vars):
    """
    rgr_era_vars: (T, F, H, W)
    No LSM masking.
    """
    x_grid = rgr_era_vars.copy()

    # same transformations as in training
    x_grid[:, ix_vs03, :, :] = np.abs(x_grid[:, ix_vs03, :, :])
    x_grid[:, ix_vs06, :, :] = np.abs(x_grid[:, ix_vs06, :, :])

    return x_grid


def save_year_output(out_npz, year, y_prob, y_bin, mean_annual, lat_out, lon_out):
    save_dict = {
        "year": int(year),
        "mean_annual": mean_annual.astype(np.float32),
        "rgrLat": lat_out.astype(np.float32),
        "rgrLon": lon_out.astype(np.float32),
        "model_vars": np.array(model_vars, dtype="U"),
        "threshold_binary": np.float32(cfg["thr_bin"]),
        "fold": np.int32(cfg["fold_to_use"]),
        "model_path": str(files["model"]),
        "region": str(cfg["region"]),
    }

    if cfg["save_daily_prob"]:
        save_dict["y_prob"] = y_prob.astype(np.float32)
    if cfg["save_daily_bin"]:
        save_dict["y_bin"] = y_bin.astype(np.float32)

    np.savez_compressed(out_npz, **save_dict)


# ============================================================
# 3) INITIAL SETUP
# ============================================================
print("Region:", cfg["region"])
print("Selected predictor indices:", sel_idx, "=>", [all_model_vars[i] for i in sel_idx])

full_dates, years_all = get_years_to_process()
print(
    "Years to process:",
    int(years_all.min()), "to", int(years_all.max()),
    "| missing:", cfg["missing_years"],
)

sample_year, sample = load_sample_predictor(years_all)
sample_vars = sample["rgrERAVarsyy"].astype(np.float32)[:, sel_idx, :, :]

lat_sample = sample.get("rgrLat", None)
lon_sample = sample.get("rgrLon", None)

if lat_sample is None or lon_sample is None:
    raise RuntimeError("Predictor .npz must contain rgrLat and rgrLon.")

lat2d, lon2d = ensure_2d_latlon(lat_sample, lon_sample)
lat_out = lat2d[:, 0]
lon_out = lon2d[0, :]

_, _, ny, nx = sample_vars.shape
print(f"Sample year {sample_year} predictor shape: {sample_vars.shape} (T, F, H, W)")
print(f"Derived grid: ny={ny}, nx={nx}")

booster = load_model(files["model"])


# ============================================================
# 4) MAIN LOOP
# ============================================================
n_done = 0
n_skip_existing = 0
n_skip_missing_pred = 0
n_skip_grid_mismatch = 0

infile_prefix = region_cfg[cfg["region"]]["infile_prefix"]
pred_prefix = region_cfg[cfg["region"]]["pred_prefix"]

for year in years_all:
    year = int(year)

    infile = paths["era_region"] / f"{infile_prefix}_{year}.npz"
    out_npz = paths["out"] / f"{pred_prefix}_{year}_fold{cfg['fold_to_use']}.npz"

    if out_npz.is_file():
        print(f"[{year}] output exists -> skip")
        n_skip_existing += 1
        continue

    if not infile.is_file():
        print(f"[{year}] predictor missing -> skip")
        n_skip_missing_pred += 1
        continue

    print(f"\n[{year}] loading predictors")
    data_tmp = np.load(infile)
    rgr_era_vars_all = data_tmp["rgrERAVarsyy"].astype(np.float32)
    rgr_era_vars = rgr_era_vars_all[:, sel_idx, :, :]

    if rgr_era_vars.shape[2:] != (ny, nx):
        print(f"[{year}] grid mismatch -> skip. Got {rgr_era_vars.shape[2:]}, expected {(ny, nx)}")
        n_skip_grid_mismatch += 1
        continue

    x_grid = prepare_predictors(rgr_era_vars)

    print(f"[{year}] predicting")
    y_prob = predict_year_chunked(
        booster=booster,
        x_grid=x_grid,
        chunk_days=cfg["chunk_days"],
    )

    y_bin = (y_prob >= cfg["thr_bin"]).astype(np.float32)
    y_bin[np.isnan(y_prob)] = np.nan

    # mean annual predicted hail frequency per grid cell
    mean_annual = np.nansum(y_bin, axis=0) / y_bin.shape[0]

    save_year_output(
        out_npz=out_npz,
        year=year,
        y_prob=y_prob,
        y_bin=y_bin,
        mean_annual=mean_annual,
        lat_out=lat_out,
        lon_out=lon_out,
    )

    print(f"[{year}] saved: {out_npz.name} | daily={y_prob.shape} | annual={mean_annual.shape}")
    n_done += 1


# ============================================================
# 5) SUMMARY
# ============================================================
print("\nDONE.")
print("  region:", cfg["region"])
print("  years predicted:", n_done)
print("  skipped (output exists):", n_skip_existing)
print("  skipped (predictor missing):", n_skip_missing_pred)
print("  skipped (grid mismatch):", n_skip_grid_mismatch)
print("  output directory:", paths["out"])

Region: EU
Selected predictor indices: [0, 1, 2, 3, 4, 6, 7, 8] => ['CAPEmax', 'SRH03', 'VS03', 'FLH', 'CINmax', 'VS06', 'DewT', 'TotalTotals']
Years to process: 1959 to 2024 | missing: [1966, 1979, 1999, 2003]
Sample year 1959 predictor shape: (365, 8, 180, 270) (T, F, H, W)
Derived grid: ny=180, nx=270
Loaded model: xgbGlobal_sigHail_FINAL_2000-2022_fold0.json

[1959] loading predictors
[1959] predicting
    chunk 000:180 done
    chunk 180:360 done
    chunk 360:365 done
[1959] saved: pred_EU_1959_fold0.npz | daily=(365, 180, 270) | annual=(180, 270)

[1960] loading predictors
[1960] predicting
    chunk 000:180 done
    chunk 180:360 done
    chunk 360:366 done
[1960] saved: pred_EU_1960_fold0.npz | daily=(366, 180, 270) | annual=(180, 270)

[1961] loading predictors
[1961] predicting
    chunk 000:180 done
    chunk 180:360 done
    chunk 360:365 done
[1961] saved: pred_EU_1961_fold0.npz | daily=(365, 180, 270) | annual=(180, 270)

[1962] loading predictors
[1962] predicting
    c

### Plot (global map)

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# GLOBAL MEAN ANNUAL FREQUENCY MAP FROM YEARLY XGBOOST OUTPUTS
#
# Reads yearly prediction files like:
#   pred_global_1959_fold2.npz
#   pred_global_1960_fold2.npz
#   ...
#
# Each file must contain:
#   - mean_annual : (lat, lon)
#
# The script:
#   1) loads the global predictor grid from one sample predictor file
#   2) loads all available yearly prediction files
#   3) reads yearly mean annual significant-hail frequency
#   4) averages across years
#   5) plots a global map
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ============================================================
# 1) CONFIG
# ============================================================
base_pred_dir = Path("/nfs/cumulus/highres_nobackup/agebhardt/preds_1959-2024_xgbGlobal")
run_date = "preds_global/fold0_dall"
pred_dir = base_pred_dir / run_date

predictor_dir = Path("/nfs/cumulus/highres_nobackup/agebhardt/e5_hailpredictors")
out_dir = Path("/cluster/home/agebhardt/plots") / run_date
out_dir.mkdir(parents=True, exist_ok=True)

cfg = {
    "fold": 0,
    "years": (1959, 2024),
    "missing_years": [1966, 1979, 1999, 2003],
    "title": "Global Mean Annual Frequency of Significant Hail (≥4.4 cm)\nxgbGlobal Final Model | 1959–2024",
    "figsize": (15, 8),
    "dpi": 200,
}


# ============================================================
# 2) HELPERS
# ============================================================
def get_available_years(start_year, end_year, missing_years):
    years = np.arange(start_year, end_year + 1, dtype=int)
    if missing_years:
        years = years[~np.isin(years, missing_years)]
    return years


def load_sample_grid(predictor_dir, years):
    sample = None
    sample_year = None

    for year in years:
        path = predictor_dir / f"ERA5_new_predictors_{year}.npz"
        if path.is_file():
            sample = np.load(path)
            sample_year = year
            break

    if sample is None:
        raise FileNotFoundError("Could not find any predictor file to infer the global grid.")

    lat = sample.get("rgrLat", None)
    lon = sample.get("rgrLon", None)

    if lat is None or lon is None:
        raise RuntimeError("Sample predictor file does not contain 'rgrLat' and 'rgrLon'.")

    lat = np.asarray(lat)
    lon = np.asarray(lon)

    if lat.ndim == 1 and lon.ndim == 1:
        lon2d = np.tile(lon[None, :], (lat.size, 1))
        lat2d = np.tile(lat[:, None], (1, lon.size))
    else:
        lat2d = lat
        lon2d = lon

    print(f"Loaded sample grid from year {sample_year}: shape = {lat2d.shape}")
    return lon2d.astype(np.float32), lat2d.astype(np.float32)


def load_yearly_mean_annual(pred_dir, year, fold):
    path = pred_dir / f"pred_global_{year}_fold{fold}.npz"
    if not path.is_file():
        return None

    data = np.load(path)

    if "mean_annual" not in data:
        raise RuntimeError(f"'mean_annual' missing in file: {path}")

    arr = data["mean_annual"].astype(np.float32)
    return arr


def load_all_yearly_mean_annual(pred_dir, years, fold):
    yearly_fields = []
    processed_years = []
    skipped_missing = []

    for year in years:
        arr = load_yearly_mean_annual(pred_dir, year, fold)

        if arr is None:
            skipped_missing.append(year)
            print(f"[{year}] prediction file missing -> skip")
            continue

        yearly_fields.append(arr)
        processed_years.append(year)

        print(f"[{year}] loaded | yearly mean shape = {arr.shape}")

    if len(yearly_fields) == 0:
        raise RuntimeError("No yearly prediction files with 'mean_annual' were found.")

    yearly_fields = np.stack(yearly_fields, axis=0)  # (n_years, lat, lon)
    return yearly_fields, processed_years, skipped_missing


def make_frequency_cmap():
    cmap = ListedColormap([
        "#f3efb4",
        "#dce9b8",
        "#b9ddb2",
        "#8fcfb5",
        "#63c0bf",
        "#4baed0",
        "#3f8fd1",
        "#3b6bc2",
        "#3247ad",
        "#1f2f88",
        "#081d58",
    ])

    bounds = [0.00, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00, 1.50, 2.00, 2.50]
    norm = BoundaryNorm(bounds, cmap.N)

    return cmap, norm, bounds


def plot_global_frequency_map(mean_annual_freq, lon2d, lat2d, title, outpath, figsize=(15, 8), dpi=200):
    cmap = ListedColormap([
        "#f3efb4",
        "#dce9b8",
        "#b9ddb2",
        "#8fcfb5",
        "#63c0bf",
        "#4baed0",
        "#3f8fd1",
        "#3b6bc2",
        "#3247ad",
        "#1f2f88",
        "#081d58",
    ])

    bounds = [0.00, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00, 1.50, 2.00, 2.50]
    norm = BoundaryNorm(bounds, cmap.N)

    data = np.ma.masked_less_equal(mean_annual_freq, 0)

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.coastlines(linewidth=0.5, zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.35, zorder=2)

    pcm = ax.pcolormesh(
        lon2d,
        lat2d,
        data,
        cmap=cmap,
        norm=norm,
        shading="auto",
        transform=ccrs.PlateCarree(),
        zorder=1,
    )

    # ax.set_title(title, fontsize=11, pad=6)

    pos = ax.get_position()
    cbar_width = pos.width * 0.8
    cbar_height = 0.015
    cbar_x = pos.x0 + (pos.width - cbar_width) / 2
    cbar_y = pos.y0 - 0.07

    cax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height])

    cbar = fig.colorbar(pcm, cax=cax, orientation="horizontal")
    cbar.set_ticks(bounds)
    cbar.set_ticklabels([
        "0.00", "0.02", "0.05", "0.10", "0.20",
        "0.50", "1.00", "1.50", "2.00", "2.50"
    ])
    cbar.set_label("Mean hail days per gridcell per year", fontsize=14)
    cbar.ax.tick_params(labelsize=14)

    fig.savefig(outpath, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 3) MAIN
# ============================================================
def main():
    years = get_available_years(
        cfg["years"][0],
        cfg["years"][1],
        cfg["missing_years"],
    )

    print(f"Requested years: {years.min()}–{years.max()}")
    print(f"Fold: {cfg['fold']}")

    lon2d, lat2d = load_sample_grid(predictor_dir, years)

    yearly_fields, processed_years, skipped_missing = load_all_yearly_mean_annual(
        pred_dir=pred_dir,
        years=years,
        fold=cfg["fold"],
    )

    mean_annual_freq = np.nanmean(yearly_fields, axis=0).astype(np.float32)
    
    # convert fraction of hail days to hail days per year
    mean_annual_freq = mean_annual_freq * 365.0

    valid = mean_annual_freq[np.isfinite(mean_annual_freq) & (mean_annual_freq > 0)]

    print("\nFrequency distribution")
    print("----------------------")
    print(f"Number of hail grid cells: {valid.size:,}")
    print(f"Mean:   {np.nanmean(valid):.4f}")
    print(f"Median: {np.nanmedian(valid):.4f}")
    print(f"P25:    {np.nanpercentile(valid, 25):.4f}")
    print(f"P75:    {np.nanpercentile(valid, 75):.4f}")
    print(f"P90:    {np.nanpercentile(valid, 90):.4f}")
    print(f"P95:    {np.nanpercentile(valid, 95):.4f}")
    print(f"P99:    {np.nanpercentile(valid, 99):.4f}")
    print(f"Max:    {np.nanmax(valid):.4f}")

    bounds = np.array([0.00, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00, 1.50, 2.00, 2.50, np.inf])

    hist, edges = np.histogram(valid, bins=bounds)
    
    print("\nMost common map frequency classes")
    print("--------------------------------")
    for i in range(len(hist)):
        print(f"{edges[i]:.2f}–{edges[i+1]:.2f}: {hist[i]:,} grid cells")
    
    imax = np.argmax(hist)
    print(
        f"\nMost frequent class: {edges[imax]:.2f}–{edges[imax+1]:.2f} "
        f"hail days per grid cell per year"
    )

    print("\nSummary")
    print("-------")
    print(f"Processed years: {len(processed_years)}")
    print(f"First/last processed year: {processed_years[0]} / {processed_years[-1]}")
    print(f"Skipped missing prediction files: {skipped_missing}")
    print(f"Mean annual frequency min/max: {np.nanmin(mean_annual_freq):.4f} / {np.nanmax(mean_annual_freq):.4f}")

    outpath = out_dir / f"global_mean_annual_sig_hail_frequency_fold{cfg['fold']}_{cfg['years'][0]}-{cfg['years'][1]}.png"

    plot_global_frequency_map(
        mean_annual_freq=mean_annual_freq,
        lon2d=lon2d,
        lat2d=lat2d,
        title=cfg["title"],
        outpath=outpath,
        figsize=cfg["figsize"],
        dpi=cfg["dpi"],
    )

    print(f"\nSaved figure to:\n{outpath}")


if __name__ == "__main__":
    main()

Requested years: 1959–2024
Fold: 0
Loaded sample grid from year 1959: shape = (721, 1440)
[1959] loaded | yearly mean shape = (721, 1440)
[1960] loaded | yearly mean shape = (721, 1440)
[1961] loaded | yearly mean shape = (721, 1440)
[1962] loaded | yearly mean shape = (721, 1440)
[1963] loaded | yearly mean shape = (721, 1440)
[1964] loaded | yearly mean shape = (721, 1440)
[1965] loaded | yearly mean shape = (721, 1440)
[1967] loaded | yearly mean shape = (721, 1440)
[1968] loaded | yearly mean shape = (721, 1440)
[1969] loaded | yearly mean shape = (721, 1440)
[1970] loaded | yearly mean shape = (721, 1440)
[1971] loaded | yearly mean shape = (721, 1440)
[1972] loaded | yearly mean shape = (721, 1440)
[1973] loaded | yearly mean shape = (721, 1440)
[1974] loaded | yearly mean shape = (721, 1440)
[1975] loaded | yearly mean shape = (721, 1440)
[1976] loaded | yearly mean shape = (721, 1440)
[1977] loaded | yearly mean shape = (721, 1440)
[1978] loaded | yearly mean shape = (721, 1440

### Plot (Regional maps)

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# REGIONAL MEAN ANNUAL FREQUENCY MAP FROM YEARLY XGBOOST OUTPUTS
#
# Reads yearly prediction files like:
#   pred_US_1959_fold2.npz
#   pred_US_1960_fold2.npz
#   ...
# or
#   pred_EU_1959_fold2.npz
#   pred_EU_1960_fold2.npz
#   ...
#
# Each file must contain:
#   - mean_annual : (lat, lon)
#
# The script:
#   1) loads the regional predictor grid from one sample predictor file
#   2) loads all available yearly prediction files
#   3) reads yearly mean annual significant-hail frequency
#   4) averages across years
#   5) plots a CONUS or Europe map
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ============================================================
# 1) CONFIG
# ============================================================
base_cumulus = Path("/nfs/cumulus/highres_nobackup/agebhardt")
base_cluster = Path("/cluster/home/agebhardt")

cfg = {
    "region": "EU",   # "US" or "EU"
    "fold": 0,
    "years": (1959, 2024),
    "missing_years": [1966, 1979, 1999, 2003],
    "figsize": (12, 7),
    "dpi": 200,
}

cities = {
    # US
    "Oklahoma City": (-97.5164, 35.4676),
    "Kansas City": (-94.5786, 39.0997),
    "Salt Lake City": (-111.8910, 40.7608),

    # Europe
    "Valencia": (-0.3763, 39.4699),
    "Paris": (2.3522, 48.8566),
    "Zurich": (8.5417, 47.3769),
    "Milan": (9.1900, 45.4642),
    "Rome": (12.4964, 41.9028),
    "Graz": (15.4395, 47.0707),
    "Belgrade": (20.4489, 44.7866),
}

region_cfg = {
    "US": {
        "pred_dir": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_US" / "fold0_dall",
        "predictor_dir": base_cumulus / "e5_hailpredictors_conus",
        "pred_prefix": "pred_US",
        "predfile_prefix": "ERA5_new_predictors_conus",
        "extent": [-127, -66, 24, 50],
        "title": "CONUS Mean Annual Frequency of Significant Hail (≥4.4 cm)\nXGBoost Global Model (Fold {cfg['fold']}) | 1959–2024",
        "outfile": "conus_mean_annual_sig_hail_frequency",
    },
    "EU": {
        "pred_dir": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_EU" / "fold0_dall",
        "predictor_dir": base_cumulus / "e5_hailpredictors_eu",
        "pred_prefix": "pred_EU",
        "predfile_prefix": "ERA5_new_predictors_eu",
        "extent": [-10, 35, 35, 65],
        "title": "Europe Mean Annual Frequency of Significant Hail (≥4.4 cm)\nXGBoost Global Model (Fold {cfg['fold']}) | 1959–2024",
        "outfile": "europe_mean_annual_sig_hail_frequency",
    },
}

if cfg["region"] not in region_cfg:
    raise ValueError(f"Unknown region {cfg['region']}. Use 'US' or 'EU'.")

pred_dir = region_cfg[cfg["region"]]["pred_dir"]
predictor_dir = region_cfg[cfg["region"]]["predictor_dir"]

out_dir = base_cluster / "plots" / "preds_1959-2024_xgbGlobal"
out_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2) HELPERS
# ============================================================
def get_available_years(start_year, end_year, missing_years):
    years = np.arange(start_year, end_year + 1, dtype=int)
    if missing_years:
        years = years[~np.isin(years, missing_years)]
    return years

def mask_below_threshold(data, lower=0.05):
    out = np.ma.masked_invalid(data)
    out = np.ma.masked_less(out, lower)
    return out


def ensure_2d_latlon(lat, lon):
    lat = np.asarray(lat)
    lon = np.asarray(lon)

    if lat.ndim == 1 and lon.ndim == 1:
        lon2d = np.tile(lon[None, :], (lat.size, 1))
        lat2d = np.tile(lat[:, None], (1, lon.size))
        return lat2d, lon2d

    if lat.ndim == 2 and lon.ndim == 2:
        return lat, lon

    raise RuntimeError(f"Unexpected lat/lon shapes: lat{lat.shape}, lon{lon.shape}")


def load_sample_grid(predictor_dir, years, predfile_prefix):
    sample = None
    sample_year = None

    for year in years:
        path = predictor_dir / f"{predfile_prefix}_{year}.npz"
        if path.is_file():
            sample = np.load(path)
            sample_year = year
            break

    if sample is None:
        raise FileNotFoundError("Could not find any predictor file to infer the regional grid.")

    lat = sample.get("rgrLat", None)
    lon = sample.get("rgrLon", None)

    if lat is None or lon is None:
        raise RuntimeError("Sample predictor file does not contain 'rgrLat' and 'rgrLon'.")

    lat2d, lon2d = ensure_2d_latlon(lat, lon)

    print(f"Loaded sample grid from year {sample_year}: shape = {lat2d.shape}")
    return lon2d.astype(np.float32), lat2d.astype(np.float32)


def load_yearly_mean_annual(pred_dir, year, fold, pred_prefix):
    path = pred_dir / f"{pred_prefix}_{year}_fold{fold}.npz"
    if not path.is_file():
        return None

    data = np.load(path)

    if "mean_annual" not in data:
        raise RuntimeError(f"'mean_annual' missing in file: {path}")

    arr = data["mean_annual"].astype(np.float32)
    return arr


def load_all_yearly_mean_annual(pred_dir, years, fold, pred_prefix):
    yearly_fields = []
    processed_years = []
    skipped_missing = []

    for year in years:
        arr = load_yearly_mean_annual(pred_dir, year, fold, pred_prefix)

        if arr is None:
            skipped_missing.append(year)
            print(f"[{year}] prediction file missing -> skip")
            continue

        yearly_fields.append(arr)
        processed_years.append(year)

        print(f"[{year}] loaded | yearly mean shape = {arr.shape}")

    if len(yearly_fields) == 0:
        raise RuntimeError("No yearly prediction files with 'mean_annual' were found.")

    yearly_fields = np.stack(yearly_fields, axis=0)  # (n_years, lat, lon)
    return yearly_fields, processed_years, skipped_missing


def make_frequency_cmap_and_norm(max_val):
    cmap = ListedColormap([
        "#f3efb4",
        "#dce9b8",
        "#b9ddb2",
        "#8fcfb5",
        "#63c0bf",
        "#4baed0",
        "#3f8fd1",
        "#3b6bc2",
        "#3247ad",
        "#1f2f88",
        "#081d58",
    ])

    base_bounds = [0.05, 0.2, 0.5, 1.0, 2.0, 4.0, 7.0]

    if max_val <= 7:
        upper = max_val
    elif max_val <= 8:
        upper = 8.0
    elif max_val <= 10:
        upper = 10.0
    elif max_val <= 12:
        upper = 12.0
    else:
        upper = float(np.ceil(max_val))

    bounds = [b for b in base_bounds if b < upper]
    bounds.append(float(upper))

    if len(bounds) < 2:
        bounds = [0.05, float(upper)]

    norm = BoundaryNorm(bounds, cmap.N, extend="max")
    return cmap, norm, bounds

def format_ticklabels(bounds):
    labels = []
    for b in bounds:
        if b < 1:
            labels.append(f"{b:.2f}")
        elif b < 10:
            labels.append(f"{b:.1f}")
        else:
            labels.append(f"{b:.1f}")
    return labels


def plot_regional_frequency_map(mean_annual_freq, lon2d, lat2d, extent, region, title, outpath, figsize=(12, 7), dpi=200):
    cmap = ListedColormap([
        "#f3efb4",
        "#dce9b8",
        "#b9ddb2",
        "#8fcfb5",
        "#63c0bf",
        "#4baed0",
        "#3f8fd1",
        "#3b6bc2",
        "#3247ad",
        "#1f2f88",
        "#081d58",
    ])
    
    bounds = [0.00, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00, 1.50, 2.00 , 2.50]
    norm = BoundaryNorm(bounds, cmap.N)

    data = np.ma.masked_less_equal(mean_annual_freq, 0)

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.coastlines(linewidth=0.5, zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.35, zorder=2)
    
    if region == "US":
        ax.add_feature(cfeature.STATES, linewidth=0.25, alpha=0.25, zorder=2)

    pcm = ax.pcolormesh(
        lon2d,
        lat2d,
        data,
        cmap=cmap,
        norm=norm,
        shading="auto",
        transform=ccrs.PlateCarree(),
        zorder=1,
    )

    for name, (lon_c, lat_c) in cities.items():
        if extent[0] <= lon_c <= extent[1] and extent[2] <= lat_c <= extent[3]:
            ax.scatter(
                lon_c, lat_c,
                s=40,
                color="black",
                edgecolor="white",
                linewidth=0.8,
                transform=ccrs.PlateCarree(),
                zorder=5,
            )
    
            ax.text(
                lon_c + 0.5, lat_c + 0.5,
                name,
                fontsize=9,
                color="black",
                transform=ccrs.PlateCarree(),
                zorder=6,
            )

    # ax.set_title(title, fontsize=11, pad=6)
    
    pos = ax.get_position()
    cbar_width = pos.width * 0.8
    cbar_height = 0.015
    cbar_x = pos.x0 + (pos.width - cbar_width) / 2
    cbar_y = pos.y0 - 0.07
    
    cax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height])
    
    cbar = fig.colorbar(pcm, cax=cax, orientation="horizontal")
    
    cbar.set_ticks(bounds)
    cbar.set_ticklabels([
        "0.00", "0.02", "0.05", "0.10",
        "0.20", "0.50", "1.00", "1.50", "2.00", "2.50",
    ])
    
    cbar.set_label("Mean hail days per gridcell per year", fontsize=14)
    cbar.ax.tick_params(labelsize=14)

    fig.savefig(outpath, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 3) MAIN
# ============================================================
def main():
    years = get_available_years(
        cfg["years"][0],
        cfg["years"][1],
        cfg["missing_years"],
    )

    reg = region_cfg[cfg["region"]]

    print(f"Region: {cfg['region']}")
    print(f"Requested years: {years.min()}–{years.max()}")
    print(f"Fold: {cfg['fold']}")

    lon2d, lat2d = load_sample_grid(
        predictor_dir=predictor_dir,
        years=years,
        predfile_prefix=reg["predfile_prefix"],
    )

    yearly_fields, processed_years, skipped_missing = load_all_yearly_mean_annual(
        pred_dir=pred_dir,
        years=years,
        fold=cfg["fold"],
        pred_prefix=reg["pred_prefix"],
    )

    mean_annual_freq = np.nanmean(yearly_fields, axis=0).astype(np.float32)

    # convert fraction of hail days to hail days per year
    mean_annual_freq = mean_annual_freq * 365.0

    valid = mean_annual_freq[np.isfinite(mean_annual_freq) & (mean_annual_freq > 0)]

    print("\nFrequency distribution")
    print("----------------------")
    print(f"Number of hail grid cells: {valid.size:,}")
    print(f"Mean:   {np.nanmean(valid):.4f}")
    print(f"Median: {np.nanmedian(valid):.4f}")
    print(f"P25:    {np.nanpercentile(valid, 25):.4f}")
    print(f"P75:    {np.nanpercentile(valid, 75):.4f}")
    print(f"P90:    {np.nanpercentile(valid, 90):.4f}")
    print(f"P95:    {np.nanpercentile(valid, 95):.4f}")
    print(f"P99:    {np.nanpercentile(valid, 99):.4f}")
    print(f"Max:    {np.nanmax(valid):.4f}")

    bounds = np.array([0.00, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00, 1.50, 2.00, 2.50, np.inf])

    hist, edges = np.histogram(valid, bins=bounds)
    
    print("\nMost common map frequency classes")
    print("--------------------------------")
    for i in range(len(hist)):
        print(f"{edges[i]:.2f}–{edges[i+1]:.2f}: {hist[i]:,} grid cells")
    
    imax = np.argmax(hist)
    print(
        f"\nMost frequent class: {edges[imax]:.2f}–{edges[imax+1]:.2f} "
        f"hail days per grid cell per year"
    )

    print("\nSummary")
    print("-------")
    print(f"Processed years: {len(processed_years)}")
    print(f"First/last processed year: {processed_years[0]} / {processed_years[-1]}")
    print(f"Skipped missing prediction files: {skipped_missing}")
    print(f"Mean annual frequency min/max: {np.nanmin(mean_annual_freq):.4f} / {np.nanmax(mean_annual_freq):.4f}")

    outpath = out_dir / f"{reg['outfile']}_fold{cfg['fold']}_{cfg['years'][0]}-{cfg['years'][1]}.png"

    plot_regional_frequency_map(
        mean_annual_freq=mean_annual_freq,
        lon2d=lon2d,
        lat2d=lat2d,
        extent=reg["extent"],
        region=cfg["region"],
        title=reg["title"],
        outpath=outpath,
        figsize=cfg["figsize"],
        dpi=cfg["dpi"],
    )

    print(f"\nSaved figure to:\n{outpath}")


if __name__ == "__main__":
    main()

Region: EU
Requested years: 1959–2024
Fold: 0
Loaded sample grid from year 1959: shape = (180, 270)
[1959] loaded | yearly mean shape = (180, 270)
[1960] loaded | yearly mean shape = (180, 270)
[1961] loaded | yearly mean shape = (180, 270)
[1962] loaded | yearly mean shape = (180, 270)
[1963] loaded | yearly mean shape = (180, 270)
[1964] loaded | yearly mean shape = (180, 270)
[1965] loaded | yearly mean shape = (180, 270)
[1967] loaded | yearly mean shape = (180, 270)
[1968] loaded | yearly mean shape = (180, 270)
[1969] loaded | yearly mean shape = (180, 270)
[1970] loaded | yearly mean shape = (180, 270)
[1971] loaded | yearly mean shape = (180, 270)
[1972] loaded | yearly mean shape = (180, 270)
[1973] loaded | yearly mean shape = (180, 270)
[1974] loaded | yearly mean shape = (180, 270)
[1975] loaded | yearly mean shape = (180, 270)
[1976] loaded | yearly mean shape = (180, 270)
[1977] loaded | yearly mean shape = (180, 270)
[1978] loaded | yearly mean shape = (180, 270)
[1980] 

### FINAL MAP (All together)

In [24]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle, ConnectionPatch
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ============================================================
# 1) CONFIG
# ============================================================
base_cumulus = Path("/nfs/cumulus/highres_nobackup/agebhardt")
base_cluster = Path("/cluster/home/agebhardt")

cfg = {
    "fold_global": 2,
    "fold_us": 2,
    "fold_eu": 2,
    "years": (1959, 2024),
    "missing_years": [1966, 1979, 1999, 2003],
    "dpi": 200,
    "figsize": (16, 9),
    "title": "Mean Annual Frequency of Significant Hail (≥4.4 cm)\nGlobal model with regional-detail insets | 1959–2024",
}

paths = {
    "pred_global": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_global",
    "pred_us": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_US",
    "pred_eu": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_EU",
    "predictor_global": base_cumulus / "e5_hailpredictors",
    "predictor_us": base_cumulus / "e5_hailpredictors_conus",
    "predictor_eu": base_cumulus / "e5_hailpredictors_eu",
    "out": base_cluster / "plots" / "preds_1959-2024_xgbGlobal",
}
paths["out"].mkdir(parents=True, exist_ok=True)

outpath = paths["out"] / (
    f"combined_global_conus_europe_mean_annual_sig_hail_"
    f"{cfg['years'][0]}-{cfg['years'][1]}_"
    f"fg{cfg['fold_global']}_fus{cfg['fold_us']}_feu{cfg['fold_eu']}.png"
)

extent_us = [-127, -66, 24, 50]
extent_eu = [-10, 35, 35, 65]


# ============================================================
# 2) HELPERS
# ============================================================
def get_available_years(start_year, end_year, missing_years):
    years = np.arange(start_year, end_year + 1, dtype=int)
    if missing_years:
        years = years[~np.isin(years, missing_years)]
    return years


def ensure_2d_latlon(lat, lon):
    lat = np.asarray(lat)
    lon = np.asarray(lon)

    if lat.ndim == 1 and lon.ndim == 1:
        lon2d = np.tile(lon[None, :], (lat.size, 1))
        lat2d = np.tile(lat[:, None], (1, lon.size))
        return lat2d, lon2d

    if lat.ndim == 2 and lon.ndim == 2:
        return lat, lon

    raise RuntimeError(f"Unexpected lat/lon shapes: lat{lat.shape}, lon{lon.shape}")


def load_sample_grid(predictor_dir, years, prefix):
    sample = None
    for year in years:
        path = predictor_dir / f"{prefix}_{year}.npz"
        if path.is_file():
            sample = np.load(path)
            break

    if sample is None:
        raise FileNotFoundError(f"Could not find sample predictor file in {predictor_dir}")

    lat = sample.get("rgrLat", None)
    lon = sample.get("rgrLon", None)

    if lat is None or lon is None:
        raise RuntimeError(f"Sample predictor file in {predictor_dir} lacks rgrLat/rgrLon")

    lat2d, lon2d = ensure_2d_latlon(lat, lon)
    return lon2d.astype(np.float32), lat2d.astype(np.float32)


def load_yearly_mean_annual(path):
    if not path.is_file():
        return None
    data = np.load(path)
    if "mean_annual" not in data:
        raise RuntimeError(f"'mean_annual' missing in file: {path}")
    return data["mean_annual"].astype(np.float32)


def load_all_yearly_mean_annual(pred_dir, years, prefix, fold):
    yearly_fields = []

    for year in years:
        path = pred_dir / f"{prefix}_{year}_fold{fold}.npz"
        arr = load_yearly_mean_annual(path)
        if arr is None:
            continue
        yearly_fields.append(arr)

    if len(yearly_fields) == 0:
        raise RuntimeError(f"No yearly prediction files found in {pred_dir}")

    yearly_fields = np.stack(yearly_fields, axis=0)
    mean_annual_freq = np.nanmean(yearly_fields, axis=0).astype(np.float32)

    # fraction of hail days -> hail days per year
    mean_annual_freq = mean_annual_freq * 365.0

    return mean_annual_freq


def make_frequency_cmap_and_norm(max_val):
    cmap = ListedColormap([
        "#f3efb4",
        "#dce9b8",
        "#b9ddb2",
        "#8fcfb5",
        "#63c0bf",
        "#4baed0",
        "#3f8fd1",
        "#3b6bc2",
        "#3247ad",
        "#1f2f88",
        "#081d58",
    ])

    base_bounds = [0.05, 0.2, 0.5, 1.0, 2.0, 4.0, 7.0]

    # choose a clean upper bound
    if max_val <= 7:
        upper = max_val
    elif max_val <= 8:
        upper = 8.0
    elif max_val <= 10:
        upper = 10.0
    elif max_val <= 12:
        upper = 12.0
    else:
        upper = float(np.ceil(max_val))

    bounds = [b for b in base_bounds if b < upper]
    bounds.append(float(upper))

    if len(bounds) < 2:
        bounds = [0.05, float(upper)]

    norm = BoundaryNorm(bounds, cmap.N, extend="max")
    return cmap, norm, bounds


def mask_below_threshold(data, lower=0.05):
    out = np.ma.masked_invalid(data)
    out = np.ma.masked_less(out, lower)
    return out


def format_ticklabels(bounds):
    labels = []
    for b in bounds:
        if b < 1:
            labels.append(f"{b:.2f}")
        elif b < 10:
            labels.append(f"{b:.1f}")
        else:
            labels.append(f"{b:.1f}")
    return labels


# ============================================================
# 3) LOAD DATA
# ============================================================
years = get_available_years(
    cfg["years"][0],
    cfg["years"][1],
    cfg["missing_years"],
)

lon_global, lat_global = load_sample_grid(
    predictor_dir=paths["predictor_global"],
    years=years,
    prefix="ERA5_new_predictors",
)
lon_us, lat_us = load_sample_grid(
    predictor_dir=paths["predictor_us"],
    years=years,
    prefix="ERA5_new_predictors_conus",
)
lon_eu, lat_eu = load_sample_grid(
    predictor_dir=paths["predictor_eu"],
    years=years,
    prefix="ERA5_new_predictors_eu",
)

freq_global = load_all_yearly_mean_annual(
    pred_dir=paths["pred_global"],
    years=years,
    prefix="pred_global",
    fold=cfg["fold_global"],
)
freq_us = load_all_yearly_mean_annual(
    pred_dir=paths["pred_us"],
    years=years,
    prefix="pred_US",
    fold=cfg["fold_us"],
)
freq_eu = load_all_yearly_mean_annual(
    pred_dir=paths["pred_eu"],
    years=years,
    prefix="pred_EU",
    fold=cfg["fold_eu"],
)

global_max = np.nanmax(freq_global)
us_max = np.nanmax(freq_us)
eu_max = np.nanmax(freq_eu)
max_val = float(np.nanmax([global_max, us_max, eu_max]))

cmap, norm, bounds = make_frequency_cmap_and_norm(max_val)

freq_global_plot = mask_below_threshold(freq_global, lower=bounds[0])
freq_us_plot = mask_below_threshold(freq_us, lower=bounds[0])
freq_eu_plot = mask_below_threshold(freq_eu, lower=bounds[0])


# ============================================================
# 4) PLOT
# ============================================================
fig = plt.figure(figsize=cfg["figsize"])

# main global map
ax_global = fig.add_axes([0.30, 0.23, 0.42, 0.60], projection=ccrs.Robinson())
# ax_global = fig.add_axes([0.25, 0.23, 0.55, 0.70], projection=ccrs.Robinson())
ax_global.set_global()

ax_global.add_feature(cfeature.LAND, facecolor="0.72", zorder=0)
ax_global.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax_global.coastlines(linewidth=0.5, zorder=2)
ax_global.add_feature(cfeature.BORDERS, linewidth=0.25, alpha=0.35, zorder=2)

pcm = ax_global.pcolormesh(
    lon_global,
    lat_global,
    freq_global_plot,
    cmap=cmap,
    norm=norm,
    shading="auto",
    transform=ccrs.PlateCarree(),
    zorder=1,
)

# boxes on global map
us_box = Rectangle(
    (extent_us[0], extent_us[2]),
    extent_us[1] - extent_us[0],
    extent_us[3] - extent_us[2],
    transform=ccrs.PlateCarree(),
    fill=False,
    edgecolor="black",
    linewidth=1.6,
    zorder=5,
)
eu_box = Rectangle(
    (extent_eu[0], extent_eu[2]),
    extent_eu[1] - extent_eu[0],
    extent_eu[3] - extent_eu[2],
    transform=ccrs.PlateCarree(),
    fill=False,
    edgecolor="black",
    linewidth=1.6,
    zorder=5,
)
ax_global.add_patch(us_box)
ax_global.add_patch(eu_box)

# CONUS inset on left side
# ax_us = fig.add_axes([0.02, 0.56, 0.27, 0.24], projection=ccrs.PlateCarree())
ax_us = fig.add_axes([0.06, 0.65, 0.27, 0.24], projection=ccrs.PlateCarree())
ax_us.set_extent(extent_us, crs=ccrs.PlateCarree())
ax_us.add_feature(cfeature.LAND, facecolor="0.85", zorder=0)
ax_us.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax_us.coastlines(linewidth=0.5, zorder=2)
ax_us.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.35, zorder=2)
ax_us.add_feature(cfeature.STATES, linewidth=0.25, alpha=0.25, zorder=2)

ax_us.pcolormesh(
    lon_us,
    lat_us,
    freq_us_plot,
    cmap=cmap,
    norm=norm,
    shading="auto",
    transform=ccrs.PlateCarree(),
    zorder=1,
)
ax_us.set_title("CONUS", fontsize=11, pad=6)

# Europe inset
ax_eu = fig.add_axes([0.68, 0.65, 0.22, 0.24], projection=ccrs.PlateCarree())
ax_eu.set_extent(extent_eu, crs=ccrs.PlateCarree())
ax_eu.add_feature(cfeature.LAND, facecolor="0.85", zorder=0)
ax_eu.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
ax_eu.coastlines(linewidth=0.5, zorder=2)
ax_eu.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.35, zorder=2)

ax_eu.pcolormesh(
    lon_eu,
    lat_eu,
    freq_eu_plot,
    cmap=cmap,
    norm=norm,
    shading="auto",
    transform=ccrs.PlateCarree(),
    zorder=1,
)
ax_eu.set_title("Europe", fontsize=11, pad=6)

# connectors
# US connectors: left corners of the global US box -> right side of US inset
for xyA, xyB in [
    ((extent_us[0], extent_us[3]), (1.00, 1.00)),  # top-left box corner -> top-right inset corner
    ((extent_us[0], extent_us[2]), (1.00, 0.00)),  # bottom-left box corner -> bottom-right inset corner
]:
    con = ConnectionPatch(
        xyA=xyA,
        coordsA=ccrs.PlateCarree()._as_mpl_transform(ax_global),
        xyB=xyB,
        coordsB=ax_us.transAxes,
        color="black",
        lw=1.5,
        zorder=10,
    )
    fig.add_artist(con)

# CONUS connectors (top-left inset)
# EU connectors: right corners of the global EU box -> left side of EU inset
for xyA, xyB in [
    ((extent_eu[1], extent_eu[3]), (0.00, 1.00)),  # top-right box corner -> top-left inset corner
    ((extent_eu[1], extent_eu[2]), (0.00, 0.00)),  # bottom-right box corner -> bottom-left inset corner
]:
    con = ConnectionPatch(
        xyA=xyA,
        coordsA=ccrs.PlateCarree()._as_mpl_transform(ax_global),
        xyB=xyB,
        coordsB=ax_eu.transAxes,
        color="black",
        lw=1.5,
        zorder=10,
    )
    fig.add_artist(con)

# thin colorbar below the global map
# get position of global axis
pos = ax_global.get_position()

# define centered colorbar below global map
cbar_width = pos.width * 0.6   # 60% of global map width
cbar_height = 0.015
cbar_x = pos.x0 + (pos.width - cbar_width) / 2
cbar_y = pos.y0 - 0.06         # vertical offset below map

cax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height])

cbar = fig.colorbar(
    pcm,
    cax=cax,
    orientation="horizontal",
    extend="max",
)

cbar.set_label("Average significant hail days per year per grid cell", fontsize=11)
cbar.set_ticks(bounds)
cbar.set_ticklabels(format_ticklabels(bounds))
cbar.ax.tick_params(labelsize=10)

# fig.suptitle(cfg["title"], fontsize=18, y=0.94)

fig.savefig(outpath, dpi=cfg["dpi"], bbox_inches="tight")
plt.close(fig)

print(f"Saved figure to:\n{outpath}")
print(f"Scale max set to data maximum: {max_val:.3f}")

Saved figure to:
/cluster/home/agebhardt/plots/preds_1959-2024_xgbGlobal/combined_global_conus_europe_mean_annual_sig_hail_1959-2024_fg2_fus2_feu2.png
Scale max set to data maximum: 3.126


## Time series